# Week 0 — Setup Check
### The Computer Vision Internship | VisionAI Lagos

All four tests must pass before Week 1.

In [ ]:
import sys, subprocess, os, warnings
warnings.filterwarnings("ignore")
def ensure(*pkgs):
    for p in pkgs:
        try: __import__(p.replace("-","_").split(".")[0])
        except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",p,"-q"])
ensure("torch","torchvision","Pillow","numpy","pandas","matplotlib","seaborn","scikit-learn","tqdm","scikit-image")
import torch, torch.nn as nn
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm
torch.manual_seed(42); np.random.seed(42)
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
MEAN    = [0.485,0.456,0.406]; STD = [0.229,0.224,0.225]
CLASSES = ["commercial","green_space","industrial","residential"]
CLS2IDX = {c:i for i,c in enumerate(CLASSES)}
IDX2CLS = {v:k for k,v in CLS2IDX.items()}
os.makedirs("outputs",exist_ok=True); os.makedirs("models",exist_ok=True)
print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | Classes: {CLASSES}")

## Test 1 — Library Versions

In [ ]:
import importlib
REQUIRED = {"torch":"torch","torchvision":"torchvision","PIL":"Pillow",
            "matplotlib":"matplotlib","seaborn":"seaborn","numpy":"numpy",
            "pandas":"pandas","tqdm":"tqdm","sklearn":"scikit-learn","skimage":"scikit-image"}
all_ok = True
for imp, pip in REQUIRED.items():
    try:
        mod = importlib.import_module(imp)
        ver = getattr(mod,"__version__","n/a")
        print(f"  ✅ {pip:<18} {ver}")
    except ImportError:
        print(f"  ❌ {pip:<18} NOT FOUND — pip install {pip}")
        all_ok = False
print(f"\nTest 1 {'PASSED ✅' if all_ok else 'FAILED ❌'}")

## Test 2 — GPU / CPU Check

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
else:
    print("  No GPU — Weeks 1-6 run fine on CPU.")
    print("  Weeks 7+ recommended on GPU (Google Colab T4 is free).")
    print("  Runtime → Change runtime type → T4 GPU")
print("\nTest 2 PASSED ✅")

## Test 3 — Generate and Verify Dataset

In [ ]:
import os, subprocess, sys

# Generate synthetic dataset if not present
DATA_DIR = 'datasets'
if not os.path.exists(DATA_DIR) or not os.listdir(DATA_DIR):
    print('Generating dataset...')
    if os.path.exists('generate_images.py'):
        subprocess.run([sys.executable, 'generate_images.py'], check=False)
    else:
        os.makedirs(DATA_DIR, exist_ok=True)
        print('⚠️  generate_images.py not found. Run from the repo root.')
        print('   git clone https://github.com/internshipinabook/cv-internship-in-a-book-in-a-book.git')
else:
    print('Dataset ready ✅')

# Verify dataset
from pathlib import Path
img_count = len(list(Path(DATA_DIR).rglob('*.png'))) + len(list(Path(DATA_DIR).rglob('*.jpg')))
print(f'Images found: {img_count:,}')


## Test 4 — Load an Image

In [ ]:
df = pd.read_csv("datasets/satellite_meta.csv")
row = df.iloc[0]
img = Image.open(f"datasets/images/{row['filename']}").convert("RGB")
arr = np.array(img)
print(f"PIL size (W,H):  {img.size}")
print(f"NumPy shape HWC: {arr.shape}")
print(f"dtype:           {arr.dtype}")
print(f"value range:     [{arr.min()}, {arr.max()}]")

import torch
from torchvision import transforms
t = transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
tensor = t(img)
print(f"\nAfter ToTensor+Normalize: {tensor.shape}  (CHW, float)")
print(f"Tensor range:             [{tensor.min():.2f}, {tensor.max():.2f}]")

# Visual grid — one per city × class
fig, axes = plt.subplots(4,4,figsize=(12,12))
for ci,lu in enumerate(["residential","commercial","industrial","green_space"]):
    for ri,city in enumerate(["Lagos","Ibadan","Kano","Abuja"]):
        r = df[(df["city"]==city)&(df["land_use"]==lu)].iloc[0]
        ax = axes[ri][ci]
        ax.imshow(np.array(Image.open(f"datasets/images/{r['filename']}")))
        if ri==0: ax.set_title(lu.replace("_"," ").title(),fontweight="bold",fontsize=9)
        if ci==0: ax.set_ylabel(city,fontsize=9)
        ax.axis("off")
plt.suptitle("Visual Inspection Grid — 4 Cities × 4 Classes",fontweight="bold",y=1.01)
plt.tight_layout()
plt.savefig("outputs/visual_grid.png",dpi=150,bbox_inches="tight",facecolor="white")
plt.show()
print("\nTest 4 PASSED ✅")
print("\n=== ALL TESTS PASSED — Ready for Week 1 ===")

## ✅ Week 0 Complete

**Before Week 1:** Open five images from each class and write a one-sentence visual description. What colour patterns are consistent? What varies?

---
*The Computer Vision Internship · VisionAI Lagos · github.com/InternshipInABook*

---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
